# CVE Data - Exploratory Data Analysis

**Purpose**: Comprehensive exploration and visualization of CVE data with enrichments

**What this notebook does**:
1. **Data Loading** - Load CVEs from database with enrichments
2. **Quality Checks** - Verify data completeness and integrity
3. **Temporal Analysis** - CVE publication trends over time
4. **Severity Analysis** - CVSS score distributions
5. **Risk Signals** - KEV, EPSS, Healthcare, ATT&CK, CHPL coverage
6. **Feature Correlations** - Relationships between risk indicators
7. **Label Distribution** - Weak label analysis

**Note**: All plots are saved externally to `outputs/plots/` to keep notebook size minimal.

---

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Force reload of modules to get latest code
import importlib
if 'src.utils.notebook_helpers' in sys.modules:
    importlib.reload(sys.modules['src.utils.notebook_helpers'])

# Import project modules
from src.core.cve_database import CVEDatabase
from src.utils.notebook_helpers import save_plot, display_sample, setup_notebook_output
from config.settings import settings

# Configure notebook display
setup_notebook_output()
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"[OK] Project root: {project_root}")
print(f"[OK] Database: {settings.get_database_path()}")
print(f"[OK] Imports successful")


## 2. Data Loading

In [ ]:
# Connect to database
db = CVEDatabase()

# Get database statistics
stats = db.get_statistics()

# Extract date range from tuple
earliest_date, latest_date = stats.get('date_range', (None, None))

print("="*70)
print("DATABASE OVERVIEW")
print("="*70)
print(f"Total CVEs: {stats['total_cves']:,}")
print(f"Date range: {earliest_date} to {latest_date}")
print("="*70)

In [ ]:
# Load CVE data with all enrichments - FULL DATASET for thesis
query = """
SELECT 
    c.cve_id,
    c.published,
    c.modified,
    c.description,
    c.cvss,
    c.cvss_vector,
    c.cwe,
    e.kev_flag,
    e.epss_score,
    e.epss_percentile,
    e.is_healthcare,
    e.healthcare_score,
    e.attack_flag,
    e.attack_technique_count,
    e.chpl_flag,
    e.is_curated,
    e.label
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.cvss IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])
df['modified'] = pd.to_datetime(df['modified'])

print(f"\n[OK] Loaded {len(df):,} CVEs (FULL DATASET)")
print(f"  Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"  Time span: {(df['published'].max() - df['published'].min()).days} days")

# Display sample
display_sample(df, n=10, title="Sample CVE Records")

## 3. Data Quality Checks

In [ ]:
print("="*70)
print("DATA QUALITY ASSESSMENT")
print("="*70)

# Completeness check
print("\n[STATS] Completeness:")
completeness = {
    'CVE ID': (df['cve_id'].notna().sum() / len(df)) * 100,
    'Published': (df['published'].notna().sum() / len(df)) * 100,
    'CVSS': (df['cvss'].notna().sum() / len(df)) * 100,
    'CWE': (df['cwe'].notna().sum() / len(df)) * 100,
    'EPSS': (df['epss_score'].notna().sum() / len(df)) * 100,
    'Description': (df['description'].notna().sum() / len(df)) * 100
}

for field, pct in completeness.items():
    status = "[OK]" if pct >= 90 else "[WARN]" if pct >= 70 else "[FAIL]"
    print(f"  {status} {field:15s}: {pct:5.1f}%")

# Enrichment signal coverage
print("\n[TARGET] Enrichment Signals:")
signals = {
    'KEV (exploited)': df['kev_flag'].sum(),
    'Healthcare-related': df['is_healthcare'].sum(),
    'ATT&CK mapped': df['attack_flag'].sum(),
    'CHPL certified': df['chpl_flag'].sum(),
    'Curated breaches': df['is_curated'].sum()
}

for signal, count in signals.items():
    pct = (count / len(df)) * 100
    print(f"  {signal:25s}: {count:6,} ({pct:5.2f}%)")

# Label distribution
if 'label' in df.columns and df['label'].notna().sum() > 0:
    print("\n  Label Distribution:")
    label_dist = df['label'].value_counts().sort_index()
    for label, count in label_dist.items():
        pct = (count / len(df)) * 100
        print(f"  Label {label}: {count:6,} ({pct:5.2f}%)")

print("\n" + "="*70)

## 4. Temporal Analysis

In [ ]:
# CVE publication trends over time
df_monthly = df.groupby(df['published'].dt.to_period('M')).size().reset_index()
df_monthly.columns = ['month', 'count']
df_monthly['month'] = df_monthly['month'].dt.to_timestamp()

fig = px.line(
    df_monthly,
    x='month',
    y='count',
    title='CVE Publications Over Time (Monthly)',
    labels={'month': 'Month', 'count': 'Number of CVEs'}
)
fig.update_traces(line_color='#2E86C1', line_width=2)
fig.update_layout(height=400, showlegend=False)

# Save externally
save_plot(fig, 'temporal_trends_monthly')

# Show summary statistics
print(f"\n Temporal Statistics:")
print(f"  Total months: {len(df_monthly)}")
print(f"  Mean CVEs/month: {df_monthly['count'].mean():.1f}")
print(f"  Peak month: {df_monthly.loc[df_monthly['count'].idxmax(), 'month'].strftime('%Y-%m')} ({df_monthly['count'].max():,} CVEs)")
print(f"  Recent 3 months avg: {df_monthly.tail(3)['count'].mean():.1f} CVEs/month")

In [ ]:
# CVE publications by year
df_yearly = df.groupby(df['published'].dt.year).agg({
    'cve_id': 'count',
    'kev_flag': 'sum',
    'is_healthcare': 'sum',
    'attack_flag': 'sum'
}).reset_index()
df_yearly.columns = ['year', 'total_cves', 'kev_cves', 'healthcare_cves', 'attack_cves']

fig = go.Figure()
fig.add_trace(go.Bar(x=df_yearly['year'], y=df_yearly['total_cves'], name='Total CVEs'))
fig.add_trace(go.Bar(x=df_yearly['year'], y=df_yearly['kev_cves'], name='KEV'))
fig.add_trace(go.Bar(x=df_yearly['year'], y=df_yearly['healthcare_cves'], name='Healthcare'))

fig.update_layout(
    title='CVE Distribution by Year with Risk Signals',
    xaxis_title='Year',
    yaxis_title='Number of CVEs',
    barmode='group',
    height=450
)

save_plot(fig, 'cve_by_year_with_signals')

print("\nYearly Summary:")
display_sample(df_yearly, n=20, title="CVEs by Year")

### 4.1 CVSS Score Temporal Trends

CVE volume growth alongside average CVSS score evolution over time.

In [ ]:
# CVE Distribution by Year with CVSS Score Trends
# Query year-by-year CVE distribution with CVSS statistics
conn = db.conn
query = """
SELECT 
    strftime('%Y', published) as year,
    COUNT(*) as total_cves,
    COUNT(cvss) as has_cvss,
    ROUND(AVG(cvss), 2) as avg_cvss
FROM cves
WHERE published IS NOT NULL
GROUP BY year
ORDER BY year
"""

df_cvss_temporal = pd.read_sql_query(query, conn)

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add CVE count bars
fig.add_trace(
    go.Bar(
        x=df_cvss_temporal['year'],
        y=df_cvss_temporal['total_cves'],
        name='Total CVEs',
        marker_color='lightblue',
        text=df_cvss_temporal['total_cves'],
        texttemplate='%{text:,}',
        textposition='outside'
    ),
    secondary_y=False
)

# Add average CVSS line
fig.add_trace(
    go.Scatter(
        x=df_cvss_temporal['year'],
        y=df_cvss_temporal['avg_cvss'],
        name='Average CVSS Score',
        mode='lines+markers',
        line=dict(color='red', width=3),
        marker=dict(size=10, symbol='diamond'),
        text=df_cvss_temporal['avg_cvss'],
        texttemplate='%{text}',
        textposition='top center'
    ),
    secondary_y=True
)

# Update axes
fig.update_xaxes(title_text="Publication Year", type='category')
fig.update_yaxes(title_text="<b>Number of CVEs</b>", secondary_y=False)
fig.update_yaxes(
    title_text="<b>Average CVSS Score</b>", 
    secondary_y=True,
    range=[0, 10]
)

# Update layout
fig.update_layout(
    title='CVE Volume Growth and CVSS Score Trends (2018-2025)',
    height=500,
    hovermode='x unified',
    showlegend=True
)

save_plot(fig, 'cvss_temporal_trends')

# Print summary statistics
print("\n[STATS] CVSS Temporal Trends:")
print(f"  Total CVEs: {df_cvss_temporal['total_cves'].sum():,}")
print(f"  Years covered: {df_cvss_temporal['year'].min()} - {df_cvss_temporal['year'].max()}")
print(f"  CVE growth: {df_cvss_temporal['total_cves'].iloc[0]:,} (2018) → {df_cvss_temporal['total_cves'].iloc[-1]:,} (2025)")
print(f"  Growth rate: {((df_cvss_temporal['total_cves'].iloc[-1] / df_cvss_temporal['total_cves'].iloc[0]) - 1) * 100:.1f}%")
print(f"\n  CVSS Score Trend:")
print(f"  2018 avg: {df_cvss_temporal['avg_cvss'].iloc[0]:.2f}")
print(f"  2025 avg: {df_cvss_temporal['avg_cvss'].iloc[-1]:.2f}")
print(f"  Change: {df_cvss_temporal['avg_cvss'].iloc[-1] - df_cvss_temporal['avg_cvss'].iloc[0]:.2f} ({((df_cvss_temporal['avg_cvss'].iloc[-1] / df_cvss_temporal['avg_cvss'].iloc[0]) - 1) * 100:.1f}%)")


## 5. CVSS Score Analysis

In [ ]:
# CVSS distribution
fig = px.histogram(
    df,
    x='cvss',
    nbins=50,
    title='CVSS Score Distribution',
    labels={'cvss': 'CVSS Score', 'count': 'Number of CVEs'},
    color_discrete_sequence=['#E74C3C']
)
fig.add_vline(x=7.0, line_dash="dash", line_color="orange", annotation_text="High (7.0)")
fig.add_vline(x=9.0, line_dash="dash", line_color="red", annotation_text="Critical (9.0)")
fig.update_layout(height=400)

save_plot(fig, 'cvss_distribution')

# CVSS statistics
print("\n[STATS] CVSS Statistics:")
print(f"  Mean: {df['cvss'].mean():.2f}")
print(f"  Median: {df['cvss'].median():.2f}")
print(f"  Std Dev: {df['cvss'].std():.2f}")
print(f"\n  Severity Distribution:")
print(f"    Low (0.0-3.9):    {((df['cvss'] < 4.0).sum()):6,} ({(df['cvss'] < 4.0).mean()*100:5.1f}%)")
print(f"    Medium (4.0-6.9): {((df['cvss'] >= 4.0) & (df['cvss'] < 7.0)).sum():6,} ({((df['cvss'] >= 4.0) & (df['cvss'] < 7.0)).mean()*100:5.1f}%)")
print(f"    High (7.0-8.9):   {((df['cvss'] >= 7.0) & (df['cvss'] < 9.0)).sum():6,} ({((df['cvss'] >= 7.0) & (df['cvss'] < 9.0)).mean()*100:5.1f}%)")
print(f"    Critical (9.0+):  {(df['cvss'] >= 9.0).sum():6,} ({(df['cvss'] >= 9.0).mean()*100:5.1f}%)")

## 6. EPSS Score Analysis

In [ ]:
# EPSS distribution (log scale)
df_epss = df[df['epss_score'].notna()].copy()

fig = px.histogram(
    df_epss,
    x='epss_score',
    nbins=100,
    title='EPSS Score Distribution (Exploit Probability)',
    labels={'epss_score': 'EPSS Score', 'count': 'Number of CVEs'},
    log_y=True,
    color_discrete_sequence=['#16A085']
)
fig.add_vline(x=0.1, line_dash="dash", line_color="orange", annotation_text="10% threshold")
fig.add_vline(x=0.5, line_dash="dash", line_color="red", annotation_text="50% threshold")
fig.update_layout(height=400)

save_plot(fig, 'epss_distribution')

# EPSS statistics
print("\n[STATS] EPSS Statistics:")
print(f"  CVEs with EPSS: {len(df_epss):,} ({len(df_epss)/len(df)*100:.1f}%)")
print(f"  Mean: {df_epss['epss_score'].mean():.4f}")
print(f"  Median: {df_epss['epss_score'].median():.4f}")
print(f"  95th percentile: {df_epss['epss_score'].quantile(0.95):.4f}")
print(f"\n  Risk Categories:")
print(f"    EPSS < 0.1:   {(df_epss['epss_score'] < 0.1).sum():6,} ({(df_epss['epss_score'] < 0.1).mean()*100:5.1f}%)")
print(f"    EPSS 0.1-0.5: {((df_epss['epss_score'] >= 0.1) & (df_epss['epss_score'] < 0.5)).sum():6,} ({((df_epss['epss_score'] >= 0.1) & (df_epss['epss_score'] < 0.5)).mean()*100:5.1f}%)")
print(f"    EPSS >= 0.5:  {(df_epss['epss_score'] >= 0.5).sum():6,} ({(df_epss['epss_score'] >= 0.5).mean()*100:5.1f}%)")

## 7. CVSS vs EPSS Relationship

In [ ]:
# Scatter plot: CVSS vs EPSS
df_plot = df[df['epss_score'].notna()].sample(min(5000, len(df_epss)))  # Sample for performance

fig = px.scatter(
    df_plot,
    x='cvss',
    y='epss_score',
    color='kev_flag',
    title='CVSS vs EPSS Score (Sample)',
    labels={'cvss': 'CVSS Score', 'epss_score': 'EPSS Score', 'kev_flag': 'KEV Listed'},
    opacity=0.6,
    color_continuous_scale='RdYlGn_r'
)
fig.update_layout(height=500)

save_plot(fig, 'cvss_vs_epss_scatter')

# Correlation
correlation = df[df['epss_score'].notna()][['cvss', 'epss_score']].corr().iloc[0, 1]
print(f"\n CVSS-EPSS Correlation: {correlation:.3f}")
print(f"   (Weak correlation suggests they measure different aspects of risk)")

## 8. Risk Signal Coverage

In [ ]:
# Multi-signal analysis
signal_data = {
    'Signal': ['KEV', 'Healthcare', 'ATT&CK', 'CHPL', 'Curated'],
    'Count': [
        df['kev_flag'].sum(),
        df['is_healthcare'].sum(),
        df['attack_flag'].sum(),
        df['chpl_flag'].sum(),
        df['is_curated'].sum()
    ]
}
signal_df = pd.DataFrame(signal_data)
signal_df['Percentage'] = (signal_df['Count'] / len(df)) * 100

fig = px.bar(
    signal_df,
    x='Signal',
    y='Count',
    title='Risk Signal Coverage',
    text='Count',
    color='Signal',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=450, showlegend=False)

save_plot(fig, 'risk_signal_coverage')

print("\n[TARGET] Risk Signal Summary:")
display_sample(signal_df, n=10)

In [ ]:
# Multi-signal overlap analysis
print("\n High-Risk CVEs (Multiple Signals):")
print(f"  KEV + Healthcare: {(df['kev_flag'] & df['is_healthcare']).sum():,}")
print(f"  KEV + ATT&CK: {(df['kev_flag'] & df['attack_flag']).sum():,}")
print(f"  KEV + CHPL: {(df['kev_flag'] & df['chpl_flag']).sum():,}")
print(f"  Healthcare + CHPL: {(df['is_healthcare'] & df['chpl_flag']).sum():,}")
print(f"\n  Triple signal (KEV + Healthcare + ATT&CK): {(df['kev_flag'] & df['is_healthcare'] & df['attack_flag']).sum():,}")

## 9. Feature Correlations

In [ ]:
# Correlation matrix for numeric features
numeric_cols = ['cvss', 'epss_score', 'epss_percentile', 'kev_flag', 
                'is_healthcare', 'attack_flag', 'chpl_flag']
corr_df = df[numeric_cols].corr()

fig = px.imshow(
    corr_df,
    text_auto='.2f',
    title='Feature Correlation Matrix',
    color_continuous_scale='RdBu_r',
    aspect='auto'
)
fig.update_layout(height=500)

save_plot(fig, 'feature_correlations')

print("\n[STATS] Key Correlations:")
# Find strongest correlations (excluding diagonal)
corr_pairs = []
for i in range(len(corr_df.columns)):
    for j in range(i+1, len(corr_df.columns)):
        corr_pairs.append((
            corr_df.columns[i],
            corr_df.columns[j],
            corr_df.iloc[i, j]
        ))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for feat1, feat2, corr_val in corr_pairs[:5]:
    print(f"  {feat1:20s} <-> {feat2:20s}: {corr_val:+.3f}")

## 10. CWE Analysis

In [ ]:
# Extract primary CWE
df['cwe_primary'] = df['cwe'].str.extract(r'(CWE-\d+)')[0]

# Top CWEs
top_cwes = df['cwe_primary'].value_counts().head(15)

fig = px.bar(
    x=top_cwes.index,
    y=top_cwes.values,
    title='Top 15 CWEs (Common Weakness Enumeration)',
    labels={'x': 'CWE', 'y': 'Number of CVEs'},
    text=top_cwes.values
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=450, showlegend=False, xaxis_tickangle=-45)

save_plot(fig, 'top_cwes')

print(f"\n[STATS] CWE Statistics:")
print(f"  CVEs with CWE: {df['cwe_primary'].notna().sum():,} ({df['cwe_primary'].notna().mean()*100:.1f}%)")
print(f"  Unique CWEs: {df['cwe_primary'].nunique():,}")
print(f"\n  Top 10 CWEs:")
for cwe, count in top_cwes.head(10).items():
    print(f"    {cwe}: {count:,}")

## 11. Label Distribution Analysis

In [ ]:
# Analyze weak label distribution
if 'label' in df.columns and df['label'].notna().sum() > 0:
    df_labeled = df[df['label'].notna()].copy()
    
    label_counts = df_labeled['label'].value_counts().sort_index()
    
    fig = px.bar(
        x=label_counts.index.astype(str),
        y=label_counts.values,
        title='Weak Label Distribution',
        labels={'x': 'Label (0=Low, 5=Critical)', 'y': 'Number of CVEs'},
        text=label_counts.values,
        color=label_counts.values,
        color_continuous_scale='YlOrRd'
    )
    fig.update_traces(texttemplate='%{text:,}', textposition='outside')
    fig.update_layout(height=400, showlegend=False)
    
    save_plot(fig, 'label_distribution')
    
    print("\n  Label Distribution:")
    for label, count in label_counts.items():
        pct = (count / len(df_labeled)) * 100
        print(f"  Label {label}: {count:6,} ({pct:5.2f}%)")
    
    # Label by signals
    print("\n  Average signals by label:")
    for label in sorted(df_labeled['label'].unique()):
        label_df = df_labeled[df_labeled['label'] == label]
        avg_kev = label_df['kev_flag'].mean()
        avg_hc = label_df['is_healthcare'].mean()
        avg_attack = label_df['attack_flag'].mean()
        print(f"    Label {label}: KEV={avg_kev:.2%}, HC={avg_hc:.2%}, ATT&CK={avg_attack:.2%}")
else:
    print("\n[WARN]  No labels found. Run Feature_Engineering notebook to generate weak labels.")

## 12. Summary Statistics

In [ ]:
print("="*70)
print("EDA SUMMARY")
print("="*70)
print(f"\n[STATS] Dataset:")
print(f"  Total CVEs: {len(df):,}")
print(f"  Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"  Avg CVEs/month: {len(df) / len(df_monthly):.1f}")

print(f"\n[WARN]  Severity:")
print(f"  Mean CVSS: {df['cvss'].mean():.2f}")
print(f"  Critical (9.0+): {(df['cvss'] >= 9.0).sum():,} ({(df['cvss'] >= 9.0).mean()*100:.1f}%)")

print(f"\n[TARGET] Risk Signals:")
print(f"  KEV: {df['kev_flag'].sum():,}")
print(f"  Healthcare: {df['is_healthcare'].sum():,}")
print(f"  ATT&CK: {df['attack_flag'].sum():,}")
print(f"  CHPL: {df['chpl_flag'].sum():,}")

print(f"\n All plots saved to: outputs/plots/")
print("="*70)

# Close database
db.conn.close()
print("\n[OK] EDA Complete")

## Next Steps

1. **Feature Engineering** -> Run `Feature_Engineering.ipynb` to create ML features
2. **Model Training** -> Run `Model_Training_And_Evaluation.ipynb` to train and compare models

---